# Info 2950 Project- Phase 3 Data Cleaning
Members:

Marianna Ruggiero
mir46@cornell.edu

Joyce Han
jh2969@cornell.edu

Fiona Chandler
fcc42@cornell.edu

Friana Engineer
fe63@cornell.edu

Research Questions: How do the technical skills that employers request differ between US data analyst, data scientist and data engineer job postings in 2023?how often are those skills mentioned in Cornells computing and data related course descriptions from the same year?

## Data Description


### Cornell API: https://classes.cornell.edu/content/SU17/api-details
The Cornell Roster API is an API that enables us to access Class Roster Data from the years 2015 to 2025.
We obtained the following attributes from the raw data:
* term: the semester in which the course was offered, such as FA21 or SP25.
* crseId: the Cornell course identifier.
* subject: the academic subject or department associated with the course.
* catalogNbr: the course catalog number.
* title: the course's title.
* description: the course description provided by Cornell.
* college: the Cornell academic group associated with the course.
* career: the academic career associated with the course.


We also created the following attributes:
* AI_narrow: indicates whether the course title or description contains "artificial intelligence," "machine learning," or the term "AI."
* AI_broad: indicates whether the course title or description contains one of the terms in our broader AI definition.


For our primary analysis we use AI_broad because it captures a wider classification of what it means to have a course that focuses on teaching students AI skills. This data was then grouped by semester and aggregated to understand the total number of courses per semester, per subject. As well as the number of courses from all courses per semester, that teach about AI. In addition to that we obtained the percentage of courses that teach AI.


The Cornell class roster provides course information for students planning their schedules. We used its API to create our dataset to answer our research question.


The raw JSON responses downloaded through the API are included in our project repository so that the dataset can be reproduced. Our processed dataset can be generated from these raw files using the Python code described below. The JSON files can also be obtained directly from using the Cornell Roster API.


### Raw Data Sources

- [Cornell raw JSON files](https://github.com/FrianaEngineer/INFO-2950-Group-Project/tree/main/phase2/raw)
- [Our World in Data raw CSV](https://ourworldindata.org/grapher/share-artificial-intelligence-job-postings.csv?v=1&csvType=full&useColumnShortNames=false)

### Our World in Data

Our World in Data: https://ourworldindata.org/grapher/share-artificial-intelligence-job-postings 

This is a data set that has pulled together data about the percentage of artificial intelligence jobs among all job postings. It allowed us to obtain the following attributes:

Entity: This is the full name of the country from which the data was gathered from 
Code: A 3 letter code that acted as a shortened name for each of the countries for graphing purposes
Year: This database has data from the years of 2014 to 2025
Share of artificial intelligence jobs among all job postings: This is a value that represents the total percentage of jobs that are considered to be AI jobs.
Columns that we changed:
In order to help with readability, we changed the column Share of artificial intelligence jobs among all job postings to Percentage. 
We sorted the rows to only include data from the US that was from 2015 onward 

This data was published by Our World in Data, using job posting data from Lightcast reported in Stanford HAI’s AI Index Report (2026). This report is produced by Stanford, which means the data is most likely trustworthy because it comes from a university research institute.

The ways that we transformed this data is by filtering our data to only contain USA information and the years between 2015 and 2025. Some of the things that could have influenced what data was collected is the wealth and resources of each country, so all of the countries that were a part of the data set were wealthy and were able to produce the reports that were needed to report on the amount of jobs that were related to AI. No people were not aware of the information that were gathered on their jobs, this is most likely due to the fact that the information was gathered about general professions and not individual jobs. 

## Data collection and cleaning

### Setting Up
We import the python libraries that are used to collect, clean, summarize, and plot the data.

In [4]:
#import libraries for data collection, cleaning & exploratory analysis
import json, re, time, pathlib
import requests
import pandas as pd
import requests
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, LogisticRegression
import duckdb
import requests
from bs4 import BeautifulSoup
import time

### Configuring Cornell data collection

We set the Cornell API address, list the semesters to collect and created a folder called raw for the downloaded JSON files. We collected FA14 through SP26 but we only used 2015–2025 in the final combined dataset.

In [5]:
#set the API address, semester list and folder for raw JSON files
BASE = "https://classes.cornell.edu/api/2.0"
TERMS = ["FA14",
         "SP15", "FA15", 
         "SP16", "FA16", 
         "SP17", "FA17", 
         "SP18", "FA18",
         "SP19", "FA19", 
         "SP20", "FA20", 
         "SP21", "FA21", 
         "SP22", "FA22",
         "SP23", "FA23", 
         "SP24", "FA24", 
         "SP25", "FA25", 
         "SP26"]
#TERMS = ["FA21"]
SLEEP = 0.25
RAW = pathlib.Path("../phase2/raw")
RAW.mkdir(exist_ok=True)

### Collect Cornell course data

For each semester- request the list of subjects and download the course records for each subject. save each response as a JSON file in the raw folder and skip files that already exist. pause between downloads and record failed requests so that we can check for missing data.

In [6]:
#download and save course records by semester and subject and record failed requests
def get_subjects(term):
    r = requests.get(f"{BASE}/config/subjects.json", params={"roster": term})
    r.raise_for_status()
    return r.json()["data"]["subjects"]

def fetch(term, subject):
    path = RAW / f"{term}_{subject}.json"
    if path.exists():
        return
    r = requests.get(f"{BASE}/search/classes.json",
                     params={"roster": term, "subject": subject})
    # The API returns 404 when a listed subject has no classes that term
    if r.status_code == 404:
        no_classes.append((term, subject))
        return
    r.raise_for_status()
    path.write_text(json.dumps(r.json()))
    time.sleep(0.5)

failed = []
no_classes = []
for term in TERMS:
    for s in get_subjects(term):
        code = s["value"]  # if this errors, print(s) and use the field that holds the subject code
        try:
            fetch(term, code)
        except Exception as e:
            failed.append((term, code, str(e)))
print("failed requests:", len(failed))
print("subjects with no classes:", no_classes)

failed requests: 0
subjects with no classes: [('FA24', 'CHERO')]


### Turning the raw files into a table

We read the saved Cornell JSON files and extracted each course’s semester, ID, subject, course number, title, description, college and academic career. then we combined these records into one table.

When a title or description was missing- replaced it with an empty string so that we could search the text for the AI keywords. We also ordered the semester labels chronologically.

In [7]:
#extract the course records from the raw JSON files and combine them into a table
rows = []
for path in sorted(RAW.glob("*.json")):
    term = path.stem.split("_", 1)[0]
    classes = json.loads(path.read_text()).get("data", {}).get("classes", [])
    for c in classes:
        rows.append({
            "term": term,
            "crseId": c.get("crseId"),
            "subject": c.get("subject"),
            "catalogNbr": c.get("catalogNbr"),
            "title": c.get("titleLong") or "",
            "description": c.get("description") or "",
            "college": c.get("acadGroup"),
            "career": c.get("acadCareer"),
        })
df = pd.DataFrame(rows)
df["term"] = pd.Categorical(df["term"], categories=TERMS, ordered=True)
print(df.groupby("term", observed=True).size())

term
FA14    4000
SP15    4088
FA15    4074
SP16    4198
FA16    4050
SP17    4228
FA17    4129
SP18    4262
FA18    4186
SP19    4272
FA19    4187
SP20    4233
FA20    4088
SP21    4231
FA21    4298
SP22    4544
FA22    4360
SP23    4659
FA23    4564
SP24    4524
FA24    4549
SP25    4697
FA25    4459
SP26    4568
dtype: int64


In [8]:
# Check the proposed Cornell course sample for Phase 3.
subjects = ["CS", "INFO", "STSCI", "ORIE"]
terms = ["SP23", "FA23"]

cornell_2023 = df.loc[
    df["term"].isin(terms) & df["subject"].isin(subjects)
].copy()

cornell_2023["has_description"] = (
    cornell_2023["description"].fillna("").str.strip().ne("")
)

course_check = cornell_2023.groupby(
    ["term", "subject"], observed=True
).agg(
    roster_rows=("has_description", "size"),
    rows_with_descriptions=("has_description", "sum"),
)

course_check

roster_rows  rows_with_descriptions
term subject                                     
SP23 CS               100                      98
     INFO              74                      74
     ORIE              45                      45
     STSCI             51                      50
FA23 CS                96                      92
     INFO              65                      65
     ORIE              58                      58
     STSCI             46                      45

In [ ]:
# Compare roster rows, course offerings, and distinct courses.
described = cornell_2023.loc[cornell_2023["has_description"]]
identified = described.dropna(subset=["crseId"])

print("Rows with descriptions:", len(described))
print("Rows missing a course ID:", len(described) - len(identified))
print("Distinct course offerings:", len(
    identified.drop_duplicates(subset=["term", "crseId"])
))
print("Distinct courses across 2023:", identified["crseId"].nunique())

Rows with descriptions: 527
Rows missing a course ID: 0
Distinct course offerings: 494
Distinct courses across 2023: 398


In [21]:
# Check whether the same course has different descriptions in the roster.
description_counts = (
    identified.assign(
        clean_description=identified["description"]
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )
    .groupby("crseId")["clean_description"]
    .nunique()
)

print("Courses with multiple descriptions:", (description_counts > 1).sum())

Courses with multiple descriptions: 7


In [22]:
# Show the roster entries for courses with multiple descriptions.
multiple_ids = description_counts[description_counts > 1].index

description_review = identified.loc[
    identified["crseId"].isin(multiple_ids),
    ["crseId", "term", "subject", "catalogNbr", "title", "description"]
].sort_values(["crseId", "term", "subject"])

with pd.option_context("display.max_colwidth", None):
    display(description_review)

,crseId,term,subject,catalogNbr,title,description
89308,352287,SP23,STSCI,4090,Theory of Statistics,"Introduction to classical theory of parametric statistical inference that builds on the material covered in BTRY 3080. Topics include sampling distributions, principles of data reduction, likelihood, parameter estimation, hypothesis testing, interval estimation, and basic asymptotic theory."
41655,352287,FA23,STSCI,4090,Theory of Statistics,"Introduction to classical theory of parametric statistical inference that builds on the material covered in BTRY 3080. Topics include: sampling distributions, principles of data reduction, likelihood, parameter estimation, hypothesis testing, interval estimation, and basic asymptotic theory."
86296,358525,SP23,CS,1112,Introduction to Computing Using MATLAB,"Programming and problem solving using MATLAB. Emphasizes the systematic development of algorithms and programs. Topics include iteration, functions, arrays, recursion, object-oriented programming, and MATLAB graphics. Assignments are designed to build an appreciation for complexity, dimension, fuzzy data, inexact arithmetic, randomness, simulation, and the role of approximation."
38667,358525,FA23,CS,1112,Introduction to Computing: An Engineering and Science Perspective,"Programming and problem solving using Python. Emphasizes the systematic development of algorithms and programs. Topics include iteration, functions, arrays, strings, recursion, object-oriented programming, algorithms, and data handling and visualization. Assignments are designed to build an appreciation for complexity, dimension, randomness, simulation, and the role of approximation in engineering and science. Weekly discussion section provides guided practice on the computer, with staff present to help. NO programming experience is necessary; some knowledge of Calculus is required."
87421,358808,SP23,INFO,2950,Introduction to Data Science,"Teaches basic mathematical methods for information science, with applications to data science. Topics include discrete probability, Bayesian methods, graph theory, power law distributions, Markov models, and hidden Markov models. Uses examples and applications from various areas of information science such as the structure of the web, genomics, social networks, natural language processing, and signal processing. Assignments require python programming.\n"
39855,358808,FA23,INFO,2950,Introduction to Data Science,"INFO 2950 is an applied introductory course on the foundations of data science, focusing on using data to identify patterns, evaluating the strength and significance of relationships, and generating predictions using data. Topics covered include the core principles of statistical programming (such as data frames, Python/R packages, reproducible workflows, and version control), univariate and multivariate statistical analysis of small and medium-size datasets, regression methods, hypothesis testing, probability models, basic supervised and unsupervised machine learning, data visualization, and network analysis. Students will learn how to use data to make effective arguments in a way that promotes the ethical usage of data. Students who complete the course will be able to produce meaningful, data-driven analyses of real-world problems and will be prepared to begin more advanced work in data-intensive domains."
89300,358957,SP23,STSCI,2130,Applied Regression Analysis,"This seven week, two-credit class will cover the regression requirements, hypothesis tests, and interpretation of results. Students will learn to identify the data necessary to perform a regression analysis, evaluate the conditions, and apply the statistical tests. Interpretation of overall results and specific predictions will be made. Independent projects by each student will be done. These will consist of identifying an issue of interest, a relevant data set, and analysis using the regression methods. These projects have ranged from movie budgets and revenues to a

In [23]:
# Keep one row per course ID and preserve every distinct description.
course_descriptions = identified[["crseId", "description"]].copy()
course_descriptions["description"] = (
    course_descriptions["description"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

courses_2023_clean = (
    course_descriptions.drop_duplicates()
    .groupby("crseId", as_index=False)["description"]
    .agg("\n---\n".join)
)

courses_2023_clean.to_csv("cornell_courses_2023_clean.csv", index=False)
print("Courses exported:", len(courses_2023_clean))

Courses exported: 398


In [25]:
# Check how many 2023 U.S. postings have skill information for each role.

jobs = pd.read_csv("data_jobs.csv")
roles = ["Data Analyst", "Data Scientist", "Data Engineer"]
posted_year = pd.to_datetime(jobs["job_posted_date"], errors="coerce").dt.year

jobs_2023 = jobs.loc[
    jobs["job_country"].eq("United States")
    & posted_year.eq(2023)
    & jobs["job_title_short"].isin(roles)
].copy()

skills = jobs_2023["job_skills"].fillna("").str.strip()
jobs_2023["has_skill_list"] = ~skills.isin(["", "[]"])

jobs_2023.groupby("job_title_short").agg(
    total_postings=("has_skill_list", "size"),
    postings_with_skills=("has_skill_list", "sum")
)

,total_postings,postings_with_skills
job_title_short,,
Data Analyst,67828,57413
Data Engineer,34583,32815
Data Scientist,58868,51686


In [26]:
# Remove rows that match across every original column.
jobs_2023_unique = jobs_2023.drop_duplicates().copy()

print("Exact duplicate rows removed:", len(jobs_2023) - len(jobs_2023_unique))

jobs_2023_unique.groupby("job_title_short").agg(
    total_postings=("has_skill_list", "size"),
    postings_with_skills=("has_skill_list", "sum")
)

Exact duplicate rows removed: 5


,total_postings,postings_with_skills
job_title_short,,
Data Analyst,67826,57411
Data Engineer,34582,32814
Data Scientist,58866,51685


In [27]:
import ast

# Keep postings with skill information.
jobs_clean = jobs_2023_unique.loc[
    jobs_2023_unique["has_skill_list"],
    ["job_title_short", "job_skills"]
].copy()

# Convert each list from text and count each skill once per posting.
def clean_skills(skill_text):
    skills = ast.literal_eval(skill_text)
    return sorted({skill.strip().lower() for skill in skills if skill.strip()})

jobs_clean["job_skills"] = jobs_clean["job_skills"].map(clean_skills)
jobs_clean = jobs_clean.loc[jobs_clean["job_skills"].map(len) > 0]

jobs_clean.to_csv("job_postings_2023_clean.csv", index=False)
print("Postings exported:", len(jobs_clean))
jobs_clean.head()

Postings exported: 141910


,job_title_short,job_skills
4,Data Analyst,"[jira, python, sql]"
6,Data Scientist,"[docker, python, r, splunk, tableau]"
7,Data Scientist,"[alteryx, cognos, python, r, spss, sql, tableau]"
14,Data Scientist,"[excel, python, r, sql, tableau]"
16,Data Engineer,"[azure, bash, databricks, excel, git, hadoop, ..."
